# PROC FACTMAC를 활용한 시설·진료과별 환자경험 평점 모델링


## 요약

여러 사이트를 운영하는 의료 시스템은 환자 만족도 점수를 좌우하는 진료 시설과 임상
진료과 간의 **상호작용 구조**를 파악하여, 저조하거나 우수한 성과를 보이는 시설/진료과
조합을 찾아내고자 합니다. 이 노트북은 `facility`와 `specialty`를 두 개의 명목형
특성 필드로, 1-10점 만족도 점수를 구간형 목표변수로 다루어 **PROC FACTMAC**으로
분해기계(factorization machine)를 적합시킵니다.

100건의 시뮬레이션 진료 사례에서 모형은 **훈련 R-제곱 0.516**(평균절대오차 0.95점,
RASE 1.25)에 도달하며, 예측된 셀 평균은 가장 강한 조합과 가장 약한 조합을 명확히
구분합니다 — `NorthClinic`/`Cardiology`가 최상위, `SouthClinic`/`Cardiology`가
최하위로, 전체 평균(약 6.8) 하나로 수렴하지 않고 내재된 상호작용을 제대로
회복합니다. (PROC FACTMAC은 ID 통과 경로에서 비-ASCII 문자를 잘라내므로,
`facility`/`specialty` 값 자체는 ASCII로 유지하고 열 머리글만 `LABEL`로
지역화했습니다.)


## 데이터 소스

모든 데이터는 DATA 스텝(`call streaminit(20240531)` + `rand()`)으로 인라인
생성되므로, 이 노트북은 완전히 자기완결적입니다 — 외부 파일이나 네트워크 접근이
없습니다. 이 환경은 라이선스가 없어 출력이 100개 관측치로 제한되므로, 설계는
**100건의 진료 사례 내에서** 분해기계를 시연하도록 규모가 정해졌습니다: 세 개
시설과 두 개 진료과의 교차(6개 셀, 셀당 평균 약 17건 — 확률적 경사하강법이 구조를
회복하기에 충분한 셀당 신호).

**WORK.ENCOUNTERS** — 100개의 합성 환자 진료 사례(진료 사례당 한 행).

| 변수 | 유형 | 역할 | 설명 |
|----------|------|------|------|
| `facility` | char(12) | 입력(명목형) | 진료 시설 — `NorthClinic`, `CentralMed`, 또는 `SouthClinic` |
| `specialty` | char(14) | 입력(명목형) | 임상 진료과 — `Cardiology` 또는 `Oncology` |
| `satisfaction` | num | 목표변수(구간형) | 1-10점 척도의 환자경험 평점으로, 시설 편향 + 진료과 편향 + 진짜 시설×진료과 상호작용 항 + 가우시안 잡음으로 생성한 뒤 [1,10]으로 절단하고 소수 첫째 자리로 반올림 |

잠재 설계에는 시설별·진료과별로 잘 분리된 편향과 강한 상호작용 효과가 내재되어
있으므로, 분해기계는 시설 단독 또는 진료과 단독 평균으로는 놓치는 구조를 회복할
수 있어야 합니다.


# PROC FACTMAC를 활용한 환자경험 평점 모델링

여러 사이트를 운영하는 의료 시스템은 다양한 **시설**과 임상 **진료과**에 걸쳐
환자 만족도 점수를 수집합니다. 시설별 또는 진료과별 평균 점수만으로는 흥미로운
신호를 놓칩니다: 어떤 진료과는 한 사이트에서는 뛰어나지만 다른 사이트에서는
부진할 수 있습니다. **분해기계(factorization machine)**는 모든 시설과 모든
진료과에 대한 잠재 요인을 학습하고, 각 평점을 전체 평균 + 특성별 효과 + 그
상호작용의 합으로 모델링하여 바로 이런 종류의 쌍별 상호작용을 포착합니다.

`PROC FACTMAC`은 확률적 경사하강법으로 이 모형을 적합시킵니다. 이 노트북에서는:

1. 100개 관측치 환경에 맞춘 규모로, 시설 x 진료과 상호작용이 내재된 합성 진료
   사례 표를 생성합니다.
2. `PROC FACTMAC`으로 분해기계를 적합시키며, 예측 점수와 반복 이력을 요청합니다.
3. 재구성 오차를 평가하고, 모형이 가장 강하거나 약하다고 표시하는 시설 x 진료과
   조합을 드러냅니다.


## 1단계 - 합성 환자경험 데이터 생성

3개 시설과 2개 진료과에 걸쳐 100건의 진료 사례를 시뮬레이션합니다. 각 시설과
진료과는 숨겨진 편향을 가지며, 특정 시설/진료과 조합이 개별 편향만으로 예측되는
것보다 높거나 낮게 평가되도록 진짜 **상호작용** 항을 추가합니다. 가우시안
잡음(표준편차 0.25)이 평점을 현실적으로 만들며, 1-10 척도로 절단하고 소수 첫째
자리로 반올림합니다. `call streaminit` 시드는 데이터를 재현 가능하게 합니다.


In [1]:
데이터 encounters;
    호출 streaminit(20240531);
    길이 facility $12 specialty $14;

    /* 시설명과 진료과 목록 (PROC FACTMAC은 ID 통과 경로에서
       비-ASCII 문자를 잘라내므로 ASCII로 유지합니다) */
    배열 facs[3] $12 _temporary_ (
        'NorthClinic' 'CentralMed' 'SouthClinic');
    배열 specs[2] $14 _temporary_ (
        'Cardiology' 'Oncology');

    /* 시설별·진료과별 잠재 평점 편향 */
    배열 f_bias[3] _temporary_ (1.5 0.0 -1.5);
    배열 s_bias[2] _temporary_ (1.0 -1.0);

    반복 enc = 1 까지 100;
        fidx = 1 + floor(3 * rand('uniform'));
        sidx = 1 + floor(2 * rand('uniform'));
        facility  = facs[fidx];
        specialty = specs[sidx];

        /* 진짜 시설 x 진료과 상호작용 항 */
        interaction = 1.2 * f_bias[fidx] * s_bias[sidx];

        satisfaction = 7.0 + f_bias[fidx] + s_bias[sidx] + interaction
            + rand('normal', 0, 0.25);

        /* 1-10 환자경험 척도 유지 */
        만약 satisfaction > 10 이면 satisfaction = 10;
        만약 satisfaction < 1  이면 satisfaction = 1;
        satisfaction = round(satisfaction, 0.1);
        출력;
    종료;

    유지 facility specialty satisfaction;
실행;



NOTE: DATA encounters


NOTE: Wrote encounters (100 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## 2단계 - 원시 평점 분포 검토

모델링에 앞서, 합성 평점이 잘 동작하는지 확인하고 시설 x 진료과 셀별 평균을
검토합니다. `PROC MEANS` 백분위수 키워드(`P25`, `MEDIAN`, `P75`)는 전체 산포를
요약하고, 두 번째 호출은 분해기계가 회복하려 시도할 상호작용을 담고 있는 셀
평균을 보여 줍니다.


In [2]:
처리 평균 데이터=encounters n mean std MIN p25 MEDIAN p75 MAX maxdec=2;
    변수 satisfaction;
    라벨 satisfaction = "환자 만족도";
실행;

처리 평균 데이터=encounters mean NWAY maxdec=2;
    분류 facility specialty;
    변수 satisfaction;
    라벨 facility = "시설"
          specialty = "진료과"
          satisfaction = "환자 만족도";
실행;


                                                  The MEANS Procedure

 Variable      Label                    N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 --------------------------------------------------------------------------------------------------------------------------------------
 satisfaction  환자 만족도                 100        6.75        1.81        4.40             5.60        6.20             8.00       10.00
 --------------------------------------------------------------------------------------------------------------------------------------

                                                  The MEANS Procedure

                                      Analysis Variable : satisfaction 환자 만족도

                                                                     N
                                      시설             진료과           Obs       Mean
                                      -------------------------------------------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3단계 - 분해기계 적합

`satisfaction`을 구간형 **목표변수**로, 두 범주형 필드 `facility`와 `specialty`를
명목형 **입력** 특성으로 모델링합니다. 주요 옵션:

- `LEARN=0.02` - 확률적 경사하강법의 스텝 크기. 이 작고 표준화된 설계에서는
  적당한 학습률이 셀 구조를 적합시키면서도 최적화 과정을 안정적으로 유지합니다.
- `L2=0.0005` - 가벼운 L2 정규화로, 요인이 전체 평균 쪽으로 과도하게 수축하지
  않도록 충분한 수준입니다.
- `SEED=20240531` - 재현 가능한 초기화.
- `OUT=scored` - 행별 예측값(`P_satisfaction`)을 기록합니다.
- `OUTSTAT=fitstats` - 반복별 RMSE 이력을 캡처하여 수렴을 확인할 수 있게 합니다.

`ID` 문은 핵심 필드를 점수화된 출력에 복사하여, 각 예측값이 시설과 진료과로
표시되도록 유지합니다.


In [3]:
처리 factmac 데이터=encounters seed=20240531 learn=0.02 l2=0.0005
        out=scored OUTSTAT=fitstats;
    입력 facility specialty / level=nominal;
    target satisfaction / level=interval;
    id facility specialty;
실행;



                         The FACTMAC Procedure

  Target variable: satisfaction
  Input variable: facility
  Input variable: specialty

Fit Statistics

  L2 Regularization              0.0005
  Learning Rate                  0.02
  Max Iterations                 100
  Number of Factors              10
  Number of Features             2
  Number of Observations         100
  Train ASE                      1.561623
  Train MAE                      0.953557
  Train R-Square                 0.515847
  Train RASE                     1.249649

Variable Importance

  Variable                            Importance
  --------                            ----------
  SPECIALTY                             0.513485
  FACILITY                              0.486515




NOTE: PROC FACTMAC data=encounters

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC FACTMAC completed.


## 4단계 - 수렴 확인

`OUTSTAT=` 표는 각 SGD 반복에서의 훈련 RMSE를 기록합니다. 단조롭게 감소하다가
평평해지는 곡선은 모형이 (기본 100회) 반복 예산 내에서 수렴했음을 알려줍니다.


In [4]:
처리 인쇄 데이터=fitstats(obs=15) 라벨 noobs;
실행;



ITERATION          RMSE
---------  ------------
1          1.6784734207
2          1.2915242083
3          1.2679925124
4          1.2654232966
5          1.2650259551
6          1.2649577538
7          1.2649457032
8          1.2649435534
9          1.2649431684
10         1.2649430993
11         1.2649430869
12         1.2649430847
13         1.2649430843
14         1.2649430842
15         1.2649430842

... 85 more observations (showing 15 of 100)




NOTE: PROC PRINT data=fitstats

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 15 observations printed, 2 variables


## 5단계 - 재구성 오차 평가

점수화된 표에는 관측된 `satisfaction`과 모형의 `P_satisfaction`이 함께 있습니다.
잔차와 절대오차를 도출한 뒤 요약합니다. 잔차가 0 근처에 중심을 두고, 예측 평점의
산포가 (전체 평균에서의 평평한 선으로 수렴하기보다) 관측된 산포에 근접한다면
분해기계가 내재된 시설 x 진료과 구조를 재현하고 있다는 뜻입니다.


In [5]:
데이터 resid;
    설정 scored;
    residual = satisfaction - p_satisfaction;
    abs_err  = abs(residual);
실행;

처리 인쇄 데이터=scored(obs=10) 라벨 noobs;
실행;

처리 평균 데이터=resid n mean std MIN p25 MEDIAN p75 MAX maxdec=3;
    변수 satisfaction p_satisfaction residual abs_err;
    라벨 satisfaction = "관찰된 만족도"
          p_satisfaction = "예측된 만족도"
          residual = "잔차"
          abs_err = "절대오차";
실행;


   FACILITY   SPECIALTY  SATISFACTION  P_SATISFACTION
-----------  ----------  ------------  --------------
SouthClinic  Oncology             6.3    6.1577265381
NorthClinic  Oncology             5.4    6.0430846516
CentralMed   Cardiology           7.9    9.5419769923
SouthClinic  Cardiology           4.7    5.8019161993
CentralMed   Oncology             6.2    5.9284427651
NorthClinic  Cardiology            10    7.6719465958
NorthClinic  Oncology             5.9    6.0430846516
NorthClinic  Cardiology            10    7.6719465958
SouthClinic  Cardiology           4.9    5.8019161993
CentralMed   Oncology             6.2    5.9284427651

... 90 more observations (showing 10 of 100)

                                                  The MEANS Procedure

 Variable        Label                       N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 --------------------------------------------------------------------------------------------


NOTE: DATA resid


NOTE: Read 100 rows from scored.
NOTE: Wrote resid (100 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 10 observations printed, 4 variables
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 6단계 - 시설 x 진료과 성과 드러내기

품질개선팀 입장에서 실행 가능한 관점은 **시설 x 진료과 조합**별 예측 평점입니다.
모형이 예측한 경험이 시스템 평균보다 훨씬 낮은 조합은 검토 후보입니다; 절대오차
열은 모형이 잘 맞는 곳과, 절단된 1-10 상한이 선형 분해기계가 도달할 수 있는
높이를 제한하는 곳을 보여 줍니다.


In [6]:
처리 평균 데이터=resid mean NWAY maxdec=3;
    분류 facility specialty;
    변수 p_satisfaction abs_err;
    라벨 facility = "시설"
          specialty = "진료과"
          p_satisfaction = "예측된 만족도"
          abs_err = "절대오차";
실행;


                                                  The MEANS Procedure

                                      Analysis Variable : p_satisfaction 예측된 만족도

                                                                     N
                                      시설             진료과           Obs       Mean
                                      -------------------------------------------
                                      CentralMed     Cardiology     13      9.542

                                                     Oncology       20      5.928

                                      NorthClinic    Cardiology     18      7.672

                                                     Oncology       16      6.043

                                      SouthClinic    Cardiology     20      5.802

                                                     Oncology       13      6.158
                                      -------------------------------------------

                                


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 결과 해석

`OUTSTAT=`의 반복 이력은 훈련 RMSE가 첫 패스에서 약 1.68이었다가 대략 일곱 번째
반복 무렵 **1.265** 근방의 평탄선에 도달하는 것을 보여 주며, 이는 SGD 적합이
반복 예산 안에서 잘 수렴했음을 확인시켜 줍니다. 적합 통계량은 **훈련 R-제곱
0.516**, **평균절대오차 0.954점**, **RASE 1.25**를 보고합니다 — 분해기계는
표준편차 1.81인 1-10 만족도 점수 분산의 약 절반을 설명하므로, 전체 평균을
예측하는 데 그치지 않고 실제로 구조를 학습하고 있는 것입니다. 잔차 요약도 이를
뒷받침합니다: 잔차 평균은 사실상 0(0.020)이며, 예측 평점은 5.80에서 9.54까지
분포합니다(표준편차 1.27) — 관측된 산포를 완전히 일치시키지는 않지만 근접하게
따라갑니다.

시설 x 진료과 표는 잠재 모형을 진료품질팀이 실행할 수 있는 것으로 바꿔 줍니다.
모형은 `CentralMed`/`Cardiology`를 최고로(예측 9.54), `SouthClinic`/`Cardiology`를
최저로(예측 5.80) 순위 매겨, Cardiology가 어떤 사이트에서는 뛰어나고 다른
사이트에서는 부진한 내재된 상호작용을 회복합니다. 절대오차 열은 모형의 한계에
대해서도 솔직합니다: 두 Oncology 셀은 거의 정확하게 재현되는 반면(평균절대오차
0.19-0.24), `NorthClinic`/`Cardiology`는 과소예측됩니다(오차 2.33) — 실제 평점이
선형 분해기계가 도달할 수 없는 절단된 1-10 상한에 몰려 있기 때문입니다.

실무자가 취할 수 있는 **다음 단계**: 과적합을 방지하기 위해 `PARTITION` 홀드아웃을
추가하거나, `LEARN=`과 `L2=`를 조정하여 편향과 분산을 절충하거나, 특성 집합을
(제공자, 방문 유형, 계절 등으로) 확장하여 분해기계가 더 높은 차수의 경험 동인을
모델링하도록 할 수 있습니다. 완전히 라이선스가 부여된 설치본에서는, 셀당 더 많은
진료 사례를 가진 더 큰 시설 x 진료과 격자를 사용하면 모형이 여기서 보여준 6-셀
설계보다 더 세밀한 상호작용 구조를 해결할 수 있을 것입니다.
